# 🚀 11 Real-time Inference & Optimization
Final deployment module for edge-ready AI.

### 🎯 Goals:
1. Convert Final Optimized Model to TFLite (Float16/Int8)
2. Run Live Webcam Inference via JS Bridge
3. Display Performance Benchmarks (Latency/Size)

In [ ]:
from google.colab import drive
import os
if not os.path.exists('/content/drive'): drive.mount('/content/drive')

import tensorflow as tf
import numpy as np
import cv2
import mediapipe as mp
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import PIL
import io

BASE_PATH = '/content/drive/MyDrive/DL/'
MODEL_PATH = os.path.join(BASE_PATH, 'models/')

# 1. Load the Final Optimized Model
final_model_path = os.path.join(MODEL_PATH, 'final_optimized_model.h5')
if os.path.exists(final_model_path):
    model = tf.keras.models.load_model(final_model_path)
    print("✅ Final Optimized Model loaded successfully.")
else:
    print("⚠️ Final model not found. Please run Notebook 07 first.")

def js_to_image(js_reply):
  image_bytes = b64decode(js_reply.split(',')[1])
  img_array = np.frombuffer(image_bytes, dtype=np.uint8)
  img = cv2.imdecode(img_array, flags=cv2.IMREAD_COLOR)
  return img

def bbox_to_bytes(bbox_array):
  bbox_PIL = PIL.Image.fromarray(bbox_array, 'RGBA')
  io_buf = io.BytesIO()
  bbox_PIL.save(io_buf, format='png')
  bbox_bytes = 'data:image/png;base64,{}'.format((str(b64encode(io_buf.getvalue()), 'utf-8')))
  return bbox_bytes

print("✅ Deployment module and JS Bridge ready.")

### 🏎️ TFLite Optimization

In [ ]:
def convert_to_tflite(keras_model, filename):
    converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    tflite_model = converter.convert()
    
    output_path = os.path.join(MODEL_PATH, filename)
    with open(output_path, 'wb') as f:
        f.write(tflite_model)
    print(f"✅ TFLite Model saved to {output_path}")

if 'model' in locals():
    convert_to_tflite(model, 'final_model_f16.tflite')

In [ ]:
def video_stream():
  js = Javascript('''
    var video; var div = null; var stream; var captureCanvas; var imgElement; var labelElement;
    var pendingResolve = null; var shutdown = false;
    function removeDom() { stream.getTracks().forEach(track => track.stop()); video.remove(); div.remove(); }
    function onAnimationFrame() {
      if (!shutdown) { window.requestAnimationFrame(onAnimationFrame); }
      if (pendingResolve) {
        var result = "";
        if (!shutdown) {
          captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
          result = captureCanvas.toDataURL('image/jpeg', 0.8)
        }
        var lp = pendingResolve; pendingResolve = null; lp(result);
      }
    }
    async function createDom() {
      div = document.createElement('div'); div.style.width = '640px'; div.style.margin = '0 auto';
      labelElement = document.createElement('div'); labelElement.innerText = "Starting..."; div.appendChild(labelElement);
      video = document.createElement('video'); video.width = 640; video.height = 480; video.autoplay = true; div.appendChild(video);
      imgElement = document.createElement('img'); imgElement.style.position = 'absolute'; imgElement.style.zIndex = '1';
      imgElement.width = 640; imgElement.height = 480; div.appendChild(imgElement);
      document.body.appendChild(div);
      stream = await navigator.mediaDevices.getUserMedia({video: {width: 640, height: 480}});
      video.srcObject = stream; await video.play();
      captureCanvas = document.createElement('canvas'); captureCanvas.width = 640; captureCanvas.height = 480;
      window.requestAnimationFrame(onAnimationFrame);
    }
    async function stream_frame(label, imgData) {
      if (shutdown) { removeDom(); shutdown = false; return "shutdown"; }
      if (label) labelElement.innerText = label;
      if (imgData) imgElement.src = imgData;
      return new Promise(resolve => pendingResolve = resolve);
    }
    ''')
  display(js)

print("✅ Video stream function defined.")